# TreeEnsemble.from_trees

This notebook shows how to build a `TreeEnsemble` from pre-extracted trees and explain it with `TreeExplainer`.

The same roundtrip conversion works for currently supported tree model families.

Each example uses this pattern:
1. Fit a model and let SHAP parse it into a `TreeEnsemble`.
2. Rebuild a new ensemble with `TreeEnsemble.from_trees(existing.trees, ...)`.
3. Explain data with the rebuilt ensemble.

In [ ]:
import warnings

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeRegressor

import shap
from shap.explainers._tree import TreeEnsemble

In [ ]:
def rebuild_with_from_trees(model, X, model_output="raw"):
    existing = TreeEnsemble(model, data=X, model_output=model_output)
    rebuilt = TreeEnsemble.from_trees(
        existing.trees,
        base_offset=existing.base_offset,
        model_output=existing.model_output,
        objective=existing.objective,
        tree_output=existing.tree_output,
        internal_dtype=existing.internal_dtype,
        input_dtype=existing.input_dtype,
        data=None,
        data_missing=None,
        fully_defined_weighting=existing.fully_defined_weighting,
        tree_limit=existing.tree_limit,
        num_stacked_models=existing.num_stacked_models,
        cat_feature_indices=existing.cat_feature_indices,
        model_type="internal",
    )
    return existing, rebuilt


def show_example(name, model, X, model_output="raw", perturbation="interventional"):
    existing, rebuilt = rebuild_with_from_trees(model, X, model_output=model_output)
    sample = X.iloc[:5] if hasattr(X, "iloc") else X[:5]
    kwargs = {}
    if perturbation == "interventional":
        kwargs = {"data": X, "feature_perturbation": "interventional"}
    else:
        kwargs = {"feature_perturbation": "tree_path_dependent"}
    e1 = shap.TreeExplainer(existing, **kwargs)(sample)
    e2 = shap.TreeExplainer(rebuilt, **kwargs)(sample)
    print(name)
    print("  predictions match:", bool((existing.predict(sample) == rebuilt.predict(sample)).all()))
    print("  values shape:", e2.values.shape)
    print("  base shape:", e2.base_values.shape)
    print("  max |delta SHAP|:", float(abs(e1.values - e2.values).max()))
    print()

In [ ]:
X, y = shap.datasets.california(n_points=200)
X_clf, y_clf = shap.datasets.iris()
y_clf = y_clf.copy()
y_clf[y_clf == 2] = 1

In [ ]:
# sklearn examples

tree_model = DecisionTreeRegressor(max_depth=3, random_state=0)
tree_model.fit(X, y)
show_example("sklearn DecisionTreeRegressor", tree_model, X)

forest = RandomForestClassifier(n_estimators=5, random_state=0)
forest.fit(X_clf, y_clf)
show_example("sklearn RandomForestClassifier", forest, X_clf)

In [ ]:
# optional backend examples (executed when package is installed)

optional_examples = []

try:
    import xgboost

    xgb = xgboost.XGBRegressor(n_estimators=5, max_depth=3, learning_rate=0.1, random_state=0)
    xgb.fit(X, y)
    optional_examples.append(("xgboost XGBRegressor", xgb, X, "raw", "interventional"))
except Exception as exc:
    warnings.warn(f"Skipping xgboost example: {exc}")

try:
    import lightgbm

    lgbm = lightgbm.LGBMRegressor(n_estimators=5, random_state=0)
    lgbm.fit(X, y)
    optional_examples.append(("lightgbm LGBMRegressor", lgbm, X, "raw", "interventional"))
except Exception as exc:
    warnings.warn(f"Skipping lightgbm example: {exc}")

try:
    import catboost

    cb = catboost.CatBoostRegressor(iterations=5, depth=3, verbose=False, random_seed=0)
    cb.fit(X, y)
    optional_examples.append(("catboost CatBoostRegressor", cb, X, "raw", "interventional"))
except Exception as exc:
    warnings.warn(f"Skipping catboost example: {exc}")

try:
    import ngboost

    ngb = ngboost.NGBRegressor(n_estimators=5, col_sample=1.0)
    ngb.fit(X, y)
    optional_examples.append(("ngboost NGBRegressor", ngb, X, 0, "interventional"))
except Exception as exc:
    warnings.warn(f"Skipping ngboost example: {exc}")

for name, model, data, output, perturbation in optional_examples:
    show_example(name, model, data, model_output=output, perturbation=perturbation)

## Deprecation note

For smaller legacy integrations (for example gpboost, ngboost, and skopt), SHAP now emits deprecation warnings and recommends exporting trees and using `TreeEnsemble.from_trees` directly.